### Nuclear mapping hg19 with bwa aln

In [ ]:
# 1) Adapter trimming
cutadapt -a AGATCGGAAGAGCACACGTCTGAACTCCAGTCA -m 30 ${sample_name}_S1_R1_001.fastq.gz > ${sample_name}-cutadapt.fastq

# 2) Mapping Nuclear
bwa aln -t 8 -n 0.01 -o 2 -l 16500 ../REF/human_g1k_v37.fasta ${sample_name}-cutadapt.fastq > ${sample_name}-nuc.sai

bwa samse -r "@RG\tID:${sample_name}\tSM:${sample_name}\tPL:Illumina" \
../REF/human_g1k_v37.fasta ${sample_name}-nuc.sai ${sample_name}-cutadapt.fastq > ${sample_name}-nuc.sam

samtools view -Sbh ${sample_name}-nuc.sam > ${sample_name}-nuc.bam

module unload java
module load java

# 3) Remove Duplicates Nuclear
samtools view -F4 -q 1 -bh ${sample_name}-nuc.bam > ${sample_name}-nuc-q1.bam
samtools sort -o ${sample_name}-nuc-q1-sorted.bam ${sample_name}-nuc-q1.bam
samtools index ${sample_name}-nuc-q1-sorted.bam

/lisc/app/java/22/bin/java -jar /lisc/app/picard/3.1.1/picard.jar MarkDuplicates 
I=${sample_name}-nuc-q1-sorted.bam 
O=${sample_name}-nuc-q1-rmdup.bam \
-MAX_FILE_HANDLES_FOR_READ_ENDS_MAP 1000 \
-REMOVE_DUPLICATES true \
-ASSUME_SORTED true \
-MAX_RECORDS_IN_RAM 1500000 \
-VALIDATION_STRINGENCY SILENT \
-METRICS_FILE metrics

/lisc/app/java/22/bin/java -jar /lisc/app/picard/3.1.1/picard.jar BuildBamIndex 
I=${sample_name}-nuc-q1-rmdup.bam \
-VALIDATION_STRINGENCY SILENT

samtools view -q 30 -bh ${sample_name}-nuc-q1-rmdup.bam > ${sample_name}-nuc-q30-rmdup.bam
samtools index ${sample_name}-nuc-q30-rmdup.bam

# 4) mapDamage analysis
mapDamage -i ${sample_name}-nuc-q30-rmdup.bam -r ../REF/human_g1k_v37.fasta -t ${sample_name}-nuc


### Mitochondrial mapping hg19 with bwa aln

In [ ]:
# 1) Adapter trimming
#cutadapt -a AGATCGGAAGAGCACACGTCTGAACTCCAGTCA -m 30 ${sample_name}_S1_R1_001.fastq.gz > ${sample_name}-cutadapt.fastq

# 2) Mapping Mitochondrial
bwa aln -t 8 -n 0.01 -o 2 -l 16500 ../calico/human.0.95.consensus.fasta ${sample_name}-cutadapt.fastq > ${sample_name}.sai

bwa samse -r "@RG\tID:${sample_name}\tSM:${sample_name}\tPL:Illumina" \
../calico/human.0.95.consensus.fasta ${sample_name}.sai ${sample_name}-cutadapt.fastq > ${sample_name}-mt.sam

samtools view -Sbh ${sample_name}-mt.sam > ${sample_name}-mt.bam

module unload java
module load java

# 3) Remove Duplicates Mitochondrial
samtools view -F4 -q 1 -bh ${sample_name}-mt.bam > ${sample_name}-mt-q1.bam
samtools sort -o ${sample_name}-mt-q1-sorted.bam ${sample_name}-mt-q1.bam
samtools index ${sample_name}-mt-q1-sorted.bam

/lisc/app/java/22/bin/java -jar /lisc/app/picard/3.1.1/picard.jar MarkDuplicates \
-I ${sample_name}-mt-q1-sorted.bam \
-O ${sample_name}-mt-q1-rmdup.bam \
-MAX_FILE_HANDLES_FOR_READ_ENDS_MAP 1000 \
-REMOVE_DUPLICATES true \
-ASSUME_SORTED true \
-MAX_RECORDS_IN_RAM 1500000 \
-VALIDATION_STRINGENCY SILENT \
-METRICS_FILE metrics

echo "index"
/lisc/app/java/22/bin/java -jar /lisc/app/picard/3.1.1/picard.jar BuildBamIndex \
-I ${sample_name}-mt-q1-rmdup.bam \
-VALIDATION_STRINGENCY SILENT

samtools view -q 30 -bh ${sample_name}-mt-q1-rmdup.bam > ${sample_name}-mt-q30-rmdup.bam
samtools index ${sample_name}-mt-q30-rmdup.bam


### Genotype calling with bcftools mpileup

In [ ]:
bcftools mpileup -Ou -f REF/human_g1k_v37.fasta {sample_name}.bam | bcftools call -m -Oz -o {sample_name}.vcf.gz
bcftools index --tbi {sample_name}.vcf.gz
bcftools mpileup -Ou -f calico/human.0.95.consensus.fasta {sample_name}-mt.bam | bcftools call -m -Oz -o {sample_name}_mt.vcf.gz
bcftools index --tbi {sample_name}_mt.vcf.gz

### Contamination estimation with calico

In [ ]:
samtools mpileup -q30 -Q30 -f human.0.95.consensus.fasta {sample_name}-mt-q30-rmdup.bam | python calico.0.2.py --maxdepth 100000 —indels

### Principal Component Analysis with ANGSD 

In [ ]:
/lisc/scratch/admixlab/aigerim/angsd/angsd -bam bamlist.txt \
      -ref REF/human_g1k_v37.fasta \
      -out angsd_output \
      -GL 2 \
      -doMajorMinor 1 \
      -doMaf 1 \
      -doGlf 2 \
      -SNP_pval 1e-6 \
      -minMapQ 30 \
      -minQ 20 \
      -nThreads 12

In [ ]:
pcangsd -b angsd_output.beagle.gz -e 2 -o pca_sgdp -t 12

### Imputation with GLIMPSE1

In [ ]:
# 1) Build a reference panel
for CHR in {1..22}; do
    # Normalize, filter, and convert to BCF
    bcftools norm -m -any ref_panel/ALL.chr${CHR}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz -Ou --threads 4 | \
    bcftools view -m 2 -M 2 -v snps --threads 4 -Ob -o ref_panel/1000GP.hg37.chr${CHR}.bcf   
    # Index the resulting BCF
    bcftools index -f ref_panel/1000GP.hg37.chr${CHR}.bcf --threads 4    
done

for i in {1..22}; do
    # convert to VCF
    bcftools view -G -m 2 -M 2 -v snps -Oz -o ref_panel/1000GP.hg37.chr${i}.vcf.gz ref_panel/1000GP.hg37.chr${i}.bcf
    bcftools index -f ref_panel/1000GP.hg37.chr${i}.vcf.gz
    # convert to TSV
    bcftools query -f'%CHROM\t%POS\t%REF,%ALT\n' ref_panel/1000GP.hg37.chr${i}.vcf.gz | bgzip -c > ref_panel/1000GP.hg37.chr${i}.tsv.gz
    tabix -s1 -b2 -e2 ref_panel/1000GP.hg37.chr${i}.tsv.gz
done

# 2) target BAM to VCF and merge
BAM={sample_name}-nuc-sorted.bam

for chr in {1..22}; do
    REFGEN=reference_genome/${chr}.fasta
    VCF=ref_panel/1000GP.hg37.chr${chr}.vcf.gz
    TSV=ref_panel/1000GP.hg37.chr${chr}.tsv.gz
    OUT=kyrgyz_vcf/{sample_name}.${chr}.vcf.gz

    bcftools mpileup -f ${REFGEN} -I -E -a 'FORMAT/DP' -T ${VCF} -r X ${BAM} -Ou | \
    bcftools call -Aim -C alleles -T ${TSV} -Oz -o ${OUT}
    bcftools index -f ${OUT}
done

for chrom in {1..22}; do
  bcftools merge -m none -r X -Oz -o merged_kyrgyz_vcf/merged.Kyrgyz.${chrom}.vcf.gz -l list_chr${chrom}.txt
done


In [ ]:
# 3) Run GLIMPSE chunk
for chrom in {1..22}; do

  bin/GLIMPSE_chunk \
    --input ref_panel/1000GP.hg37.chr${chrom}.vcf.gz \
    --region X \
    --window-size 1000000 \
    --buffer-size 200000 \
    --output chunks.${chrom}.txt

done

# 4) Run GLIMPSE phase
VCF="merged_kyrgyz_vcf/merged.Kyrgyz.${chrom}.vcf.gz"
REF="ref_panel/1000GP.hg37.chr${chrom}.bcf"
MAP="../maps/genetic_maps.b37/chr${chrom}.b37.gmap.gz"
CHUNKS="chunks.${chrom}.txt"

while IFS="" read -r LINE || [ -n "$LINE" ]; do
  printf -v ID "%02d" $(echo $LINE | cut -d" " -f1)
  IRG=$(echo $LINE | cut -d" " -f3)
  ORG=$(echo $LINE | cut -d" " -f4)
  OUT="GLIMPSE_impute/merged.Kyrgyz.${chrom}.imputed.${ID}.bcf"

  bin/GLIMPSE_phase \
    --input ${VCF} \
    --reference ${REF} \
    --map ${MAP} \
    --main 15 \
    --input-region ${IRG} \
    --output-region ${ORG} \
    --output ${OUT} 2>> errors.chr${chrom}.log

done < ${CHUNKS}

# 5) Run GLIMPSE ligate
LST="list.chr${CHR}.txt"
ls merged.Kyrgyz.${CHR}.imputed.*.bcf > ${LST}
OUT="merged.Kyrgyz.${CHR}.merged.bcf"
bin/GLIMPSE_ligate --input ${LST} --output ${OUT}

bcftools index -f ${OUT}

# 6) Run GLIMPSE sample
VCF="merged.Kyrgyz.${CHR}.merged.bcf"
OUT="merged.Kyrgyz.${CHR}.phased.bcf"
bin/GLIMPSE_sample --input ${VCF} --solve --output ${OUT}

bcftools index -f ${OUT}


### Relatedness with ancIBD

In [ ]:
bcftools annotate -a merged.Kyrgyz.${CHR}.merged.bcf -c FORMAT/GP merged.Kyrgyz.${CHR}.phased.bcf -Oz -o merged.Kyrgyz.${CHR}.combined.vcf

In [ ]:
# VCF to hdf5
chs = range(1,23)
for ch in chs:
    #base_path = f"/lisc/scratch/admixlab/aigerim"
    vcf_to_1240K_hdf(in_vcf_path = "{base_path}/GLIMPSE/tutorial/merged.Kyrgyz.{ch}.combined.vcf",
                     path_vcf = "{base_path}/ancIBD/vcf/merged.Kyrgyz.1240.{ch}.vcf",
                     path_h5 = "{base_path}/ancIBD/data/hdf5/merged.Kyrgyz.{ch}.h5",
                     marker_path = "{base_path}/ancIBD/data/filters/snps_bcftools_ch{ch}.csv",
                     map_path = "{base_path}/ancIBD/data/map/v51.1_1240k.snp",
                     col_sample_af = "",
                     buffer_size=20000, chunk_width=8, chunk_length=20000,
                     ch=ch)

In [ ]:
# Calling IBD on autosomes
from ancIBD.run import hapBLOCK_chroms

for ch in range(1,23):
    df_ibd = hapBLOCK_chroms(folder_in='./data/hdf5/merged.Kyrgyz.',
                             iids=iids, run_iids=[],
                             ch=ch, folder_out='./output/',
                             output=False, prefix_out='', logfile=False,
                             l_model='h5', e_model='haploid_gl2', h_model='FiveStateScaled', t_model='standard',
                             p_col='variants/RAF',
                             ibd_in=1, ibd_out=10, ibd_jump=400,
                             min_cm=6, cutoff_post=0.99, max_gap=0.0075)
# Combine
from ancIBD.IO.ind_ibd import combine_all_chroms
combine_all_chroms(chs=range(1,23),
                   folder_base='./output/ch',
                   path_save='./output/ch_all.tsv')

# Summary table
from ancIBD.IO.ind_ibd import create_ind_ibd_df
df_res = create_ind_ibd_df(ibd_data = './output/ch_all.tsv',
                      min_cms = [8, 12, 16, 20], snp_cm = 220, min_cm = 5, sort_col = 0,
                      savepath = "./output/Kyrgyz.tsv")

In [ ]:
# X chromosome
!ancIBDX --vcf /lisc/scratch/admixlab/aigerim/GLIMPSE/tutorial/merged.Kyrgyz.23.combined.vcf --ch X \
        --marker_path ancIBDX/snps_bcftools_chX_1240k.csv \
        --map_path ancIBDX/v51.1_1240k.chrX.map \
        --af_column variants/RAF --ploidy ancIBDX/ploidy.txt --min 8 --bin 8,12,16,20

### Haplogroup assignment with haploGrouper

In [ ]:
# mtDNA
python haploGrouper.py -v /lisc/project/admixlab/AR_Kyrgyz/{sample_name}_mt.vcf.gz
-t data/chrMT_phylotree17_tree.txt     
-l data/chrMT_phylotree17_loci.txt     
-f data/rCRS.fasta     
-o {sample_name}_haplogroup.txt     
-x docs/chrMT_HG00096_allScores.txt

In [ ]:
# Y chromosome
python haploGrouper.py -v {sample_name}_y.vcf.gz     
-t data/chrY_isogg2019_tree.txt     
-l data/chrY_isogg2019-decode1_loci_b37.txt     
-o docs/{sample_name}_y_haplogroup.txt     
-x docs/chrY_isogg2019-decode1_HG00096_allScores.txt

### Runs of homozygosity with hapROH

In [ ]:
# run hapROH on each chromosome for each individual
from hapsburg.PackagesSupport.hapsburg_run import hapsb_ind
hapsb_ind(iid="{sample_name}", chs=range(1, 23), 
          path_targets='merged.Kyrgyz.all_chromosomes', # The path before the .ind, .snp, .geno
          h5_path1000g="1000g1240khdf5/all1240/chr", 
          meta_path_ref="1000g1240khdf5/all1240/meta_df_all.csv", 
          folder_out='Desktop/Kyrgyz/',
          processes=6, output=True,
          readcounts=False, logfile=True, combine=True)

In [ ]:
# create a csv file
iids = [{sample_name}, {sample_name}, {sample_name}]
df = pd.DataFrame({"iid":iids})
df["clst"] = "Kyrgyz"
df.to_csv("Desktop/Kyrgyz/meta_blank.csv", 
          sep=",", index=False)

In [ ]:
# combine roh.csv files into one csv
import pandas as pd
df_meta = pd.read_csv("Desktop/Kyrgyz/meta_blank.csv", sep=",", dtype={"iid": str})
df1 = pp_individual_roh(iids, meta_path="Desktop/Kyrgyz/meta_blank.csv", 
                        base_folder="Desktop/Kyrgyz/",
                        save_path="Desktop/Kyrgyz/combined_roh05.csv", 
                        output=False, min_cm=[4, 8, 12, 20], snp_cm=50, 
                        gap=0.5, min_len1=2.0, min_len2=4.0)

df1["iid"] = df1["iid"].astype(str)
df_meta["iid"] = df_meta["iid"].astype(str)
df1 = pd.merge(df1, df_meta, on="iid")

In [ ]:
# plot ROH
from hapsburg.figures.plot_individual_roh import plot_roh_individual
plot_roh_individual(iid="{sample_name}", folder="Desktop/Kyrgyz/", 
                    prefix_out="", min_cm=4, plot_bad=False, savepath="")  

### Principal Component Analysis with smartsnp

In [ ]:
#extract BA and IA Central Eurasian populations from AADR dataset v62.0_1240k_public into AADR.pca
./plink2 --bfile v62.0_1240k_public --keep inds.txt --make-bed --out AADR.pca
#make bed/bim/fam files out of vcf
./plink2 --vcf merged.Kyrgyz.vcf.gz --make-bed --out merged.Kyrgyz
#subset Kyrgyz files to the AADR sites
awk 'BEGIN{OFS="\t"} {print $1, $4-1, $4, $2}' AADR.pca.bim > pca.sites.bed
./plink2 \
  --bfile merged.Kyrgyz \
  --extract range pca.sites.bed \
  --make-bed \
  --out Kyrgyz.pca
#using convertf make geno/snp/ind files out of AADR.pca and Kyrgyz.pca
convertf -p par.PED2EIGENSTRAT
#using mergeit merge two sets of geno/snp/ind files into AADR_Kyrgyz.geno/snp/ind
mergeit -p mergeit.params

### qpAdm

In [ ]:
qpadm -p parqpadm.txt > Kyrgyz.log

In [ ]:
parqpadm.txt

genotypename: ../popgen/complete.AADR.Kyrgyz.geno
snpname: ../popgen/complete.AADR.Kyrgyz.snp
indivname: ../popgen/complete.AADR.Kyrgyz.ind
popleft: left.pops
popright: right.pops
details: YES
maxrank: 7
inbreed: NO

### ADMIXTURE

In [ ]:
for i in {2..10}; do admixture --cv Kyrgyz.pca.bed $i > log${i}.out; done